# RAG Knowledge Ingestion

Vectoriza los PDFs de `mocks/knowledge/` y los inserta en la tabla `documents` de Postgres (pgvector).

In [1]:
import os
import pathlib

import psycopg
import tiktoken
import yaml
from openai import OpenAI
from pypdf import PdfReader

## Configuración

In [2]:
CONFIG_PATH = pathlib.Path("../core/config.yml")
KNOWLEDGE_DIR = pathlib.Path("../mocks/knowledge")

_cfg = yaml.safe_load(CONFIG_PATH.read_text())

pg = _cfg["services"]["postgres"]
PG_CONNSTR = f"host={pg['host']} port={pg['port']} user={pg['user']} password={pg['password']} dbname={pg['database']}"

OPENAI_API_KEY = _cfg.get("rag", {}).get("api_key") or os.environ.get("OPENAI_API_KEY", "")
EMBEDDING_MODEL = _cfg.get("rag", {}).get("embedding_model", "text-embedding-3-small")

CHUNK_TOKENS = 500
OVERLAP_TOKENS = 100

assert OPENAI_API_KEY, "Falta OPENAI_API_KEY"
assert KNOWLEDGE_DIR.exists(), f"Directorio no encontrado: {KNOWLEDGE_DIR.resolve()}"

print(f"Modelo embedding: {EMBEDDING_MODEL}")
print(f"Directorio docs: {KNOWLEDGE_DIR.resolve()}")
print(f"Chunk: {CHUNK_TOKENS} / {OVERLAP_TOKENS} tokens")

Modelo embedding: text-embedding-3-small
Directorio docs: C:\Users\Th4nos\Documents\github\msc-muia-2026\mocks\knowledge
Chunk: 500 / 100 tokens


## Extracción de texto de PDFs

In [3]:
def extract_pdf_text(path: pathlib.Path) -> str:
    reader = PdfReader(path)
    pages = (page.extract_text() or "" for page in reader.pages)
    return "\n\n".join(p.replace("\x00", "") for p in pages)


pdfs = sorted(KNOWLEDGE_DIR.glob("*.pdf"))
print(f"{len(pdfs)} PDFs encontrados")
for p in pdfs:
    print(f"  {p.name}")

4 PDFs encontrados
  iea_pvps_analytical_monitoring.pdf
  iea_pvps_degradation_failure.pdf
  pv_underperformance_causes.pdf
  realtime_anomaly_detection.pdf


## Chunking por tamaño fijo

In [4]:
enc = tiktoken.encoding_for_model("text-embedding-3-small")


def chunk_text(text: str, chunk_tokens: int, overlap_tokens: int) -> list[str]:
    tokens = enc.encode(text)
    chunks = []
    step = chunk_tokens - overlap_tokens
    for start in range(0, len(tokens), step):
        chunk = tokens[start : start + chunk_tokens]
        if len(chunk) < 20:
            break
        chunks.append(enc.decode(chunk))
    return chunks


# (title, source, snippet)
all_chunks: list[tuple[str, str, str]] = []

for pdf in pdfs:
    text = extract_pdf_text(pdf)
    chunks = chunk_text(text, CHUNK_TOKENS, OVERLAP_TOKENS)
    print(f"{pdf.name}: {len(text):,} chars → {len(chunks)} chunks")
    for chunk in chunks:
        all_chunks.append((pdf.stem, pdf.name, chunk))

print(f"\nTotal chunks: {len(all_chunks)}")

iea_pvps_analytical_monitoring.pdf: 162,221 chars → 95 chunks


iea_pvps_degradation_failure.pdf: 139,676 chars → 83 chunks


pv_underperformance_causes.pdf: 128,985 chars → 87 chunks


realtime_anomaly_detection.pdf: 64,947 chars → 36 chunks

Total chunks: 301


## Embeddings (batch OpenAI)

In [5]:
client = OpenAI(api_key=OPENAI_API_KEY)

BATCH_SIZE = 100


def embed_batch(texts: list[str]) -> list[list[float]]:
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [item.embedding for item in response.data]


snippets = [c[2] for c in all_chunks]
embeddings: list[list[float]] = []

for i in range(0, len(snippets), BATCH_SIZE):
    batch = snippets[i : i + BATCH_SIZE]
    embeddings.extend(embed_batch(batch))
    print(f"- Embeddings: {min(i + BATCH_SIZE, len(snippets))}/{len(snippets)}")

print(f"Dimensión embedding: {len(embeddings[0])}")

- Embeddings: 100/301


- Embeddings: 200/301


- Embeddings: 300/301
- Embeddings: 301/301
Dimensión embedding: 1536


## Inserción en postgres

In [6]:
INSERT_SQL = """
    INSERT INTO documents (title, content, embedding, source)
    VALUES (%s, %s, %s, %s)
"""

rows = [
    (title, snippet, embedding, source)
    for (title, source, snippet), embedding in zip(all_chunks, embeddings)
]

with psycopg.connect(PG_CONNSTR) as conn:
    with conn.cursor() as cur:
        cur.executemany(INSERT_SQL, rows)
    conn.commit()

print(f"{len(rows)} filas insertadas en la tabla documents")

301 filas insertadas en la tabla documents


## Verificación

In [7]:
with psycopg.connect(PG_CONNSTR) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT source, COUNT(*) FROM documents GROUP BY source ORDER BY source")
        rows_check = cur.fetchall()

print("Documentos en BD:")
for source, count in rows_check:
    print(f"  {source}: {count} chunks")

Documentos en BD:
  iea_pvps_analytical_monitoring.pdf: 95 chunks
  iea_pvps_degradation_failure.pdf: 83 chunks
  pv_underperformance_causes.pdf: 87 chunks
  realtime_anomaly_detection.pdf: 36 chunks
